In [ ]:
import numpy as np
import time
import pandas as pd
from collections import deque


tol = 1e-5
eta = 0.1
delta_max = 1.25


In [ ]:
def apGrad(f,x,):
  n=len(x)
  grad = np.zeros(n)

  eps=np.finfo(float).eps
  for i in range(n):
    h=np.zeros(n)
    h[i]=eps**(1.0/3)*(abs(x[i])+1)
    grad[i]=0.5*(f(x+h)-f(x-h))/h[i]

  return (grad)

In [ ]:
def apHess(f, x):
    n = len(x)
    hess = np.zeros((n, n))
    eps = np.finfo(float).eps
    for i in range(n):
        hi = np.zeros(n)
        hi[i] = eps**(0.25) * (abs(x[i]) + 1)
        for j in range(n):
            hj = np.zeros(n)
            hj[j] = eps**(0.25) * (abs(x[j]) + 1)
            hess[i, j] = 0.25*(f(x+hi+hj)-f(x-hi+hj)-f(x+hi-hj)+f(x-hi-hj))/(hi[i]/hj[j])
    return hess

Iniciamos definiendo la funcion Dogleg

In [ ]:
def pDogLeg(B, g, delta, tol=1e-5):
    """
    Calcula el paso Dog-Leg para una región de confianza.

    Args:
        B (ndarray): Matriz Hessiana aproximada (debe ser simétrica y positiva definida).
        g (ndarray): Vector gradiente en el punto actual.
        delta (float): Radio actual de la región de confianza.
        tol (float): Tolerancia para cálculos numéricos.

    Returns:
        ndarray: Vector de paso p dentro de la región de confianza.
    """
    # Verificar que g no sea cero para evitar divisiones por cero
    norm_g = np.linalg.norm(g)
    if norm_g < tol:
        return np.zeros_like(g)  # Si g ≈ 0, estamos cerca de un punto estacionario

    # 1. Calcular α_u (escalar de Cauchy) como se muestra en el diagrama de clase
    numerador = g.T @ g
    denominador = g.T @ B @ g

    # Verificar que el denominador no sea cercano a cero y sea positivo
    if denominador > tol:
        alpha_u = numerador / denominador
    else:
        # Si denominador ≈ 0 o negativo, usar una aproximación segura
        # Esto garantiza una dirección de descenso
        alpha_u = 1.0 / (np.linalg.norm(B) + tol)

    # 2. Verificar si el paso de Cauchy está fuera de la región de confianza
    if alpha_u * norm_g >= delta:
        # Paso de Cauchy escalado al borde de la región de confianza
        p = -delta * g / norm_g
        return p

    # 3. Calcular paso de Newton p_B = -B⁻¹g
    try:
        # Intentar resolver el sistema lineal B·p_B = -g
        p_B = -np.linalg.solve(B, g)

        # Verificar que la solución sea correcta (B·p_B ≈ -g)
        residual = np.linalg.norm(B @ p_B + g) / (norm_g + tol)
        if residual > 1e-2:  # Solución no confiable
            raise np.linalg.LinAlgError("Solución numérica imprecisa")

    except np.linalg.LinAlgError:
        # Si B no es invertible o la solución es imprecisa
        raise ValueError("B no es invertible ")

        # Verificar que p_B sea dirección de descenso (g·p_B < 0)
        if g @ p_B > 0:
            # Si no es dirección de descenso, usar el paso de Cauchy
            p_B = -alpha_u * g

    # 4. Verificar si el paso de Newton está dentro de la región de confianza
    norm_pB = np.linalg.norm(p_B)
    if norm_pB <= delta:
        return p_B

    # 5. Calcular paso de Cauchy p_u
    p_u = -alpha_u * g

    # 6. Resolver para alpha_* que satisfaga ||p_u + α_*(p_B - p_u)||² = Δ²
    d = p_B - p_u
    norm_d_squared = np.dot(d, d)

    if norm_d_squared < tol:
        # Si p_B ≈ p_u, usar paso de Cauchy escalado
        return -delta * g / norm_g

    a = norm_d_squared
    b = 2 * np.dot(p_u, d)
    c = np.dot(p_u, p_u) - delta**2

    # Resolver ecuación cuadrática
    discriminante = b**2 - 4*a*c
    if discriminante < 0:
        # En caso de problemas numéricos, ajustar el discriminante
        discriminante = 0

    alpha_star = (-b + np.sqrt(discriminante)) / (2*a)
    # Asegurar que alpha_star esté en (0, 1)
    alpha_star = max(0, min(1, alpha_star))

    # 7. Calcular el paso final (combinación de Cauchy y Newton)
    p = p_u + alpha_star * d

    # Verificación final: asegurar que ||p|| ≤ delta
    norm_p = np.linalg.norm(p)
    if norm_p > delta * (1 + tol):
        p = p * (delta / norm_p)

    return p

definir funcion de region de confianza dogleg

In [ ]:
def mRCHess(f, x0, itmax):
    """
    Método de Región de Confianza con paso Dog-Leg y Hessiana aproximada.

    Args:
        f (callable): función a minimizar, f(x: ndarray) -> float
        x0 (ndarray): punto inicial
        itmax (int): máximo de iteraciones

    Returns:
        xk (ndarray): aproximación final
        k (int): número de iteraciones realizadas
    """
    tol, eta = 1e-5, 0.1
    delta_max, delta = 1.25, 1.0

    xk = x0.astype(float).copy()
    gk = apGrad(f, xk)
    its = 0

    while np.linalg.norm(gk, ord=np.inf) > tol and its < itmax:
        Bk = apHess(f, xk)
        pk = pDogLeg(Bk, gk, delta)

        f_k    = f(xk)
        f_new  = f(xk + pk)
        red_r  = f_k - f_new
        red_p  = - (gk @ pk) - 0.5 * (pk @ Bk @ pk)
        rho    = red_r / red_p if red_p > 0 else 0.0

        # ajustar región de confianza
        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and np.linalg.norm(pk) >= delta:
            delta = min(2 * delta, delta_max)

        # aceptar paso
        if rho > eta:
            xk = xk + pk
            gk = apGrad(f, xk)

        its += 1

    return xk, its


Metodo BFGS

In [ ]:
def rcBFGS(f, x0, itmax):
    """
    Región de Confianza con BFGS para actualizar el modelo.

    Args:
        f (callable): función escalar a minimizar.
        x0 (ndarray): punto inicial.
        itmax (int): iteraciones máximas.

    Returns:
        xk (ndarray): aproximación final.
        k  (int): número de iteraciones realizadas.
    """
    tol   = 1e-5
    eta   = 0.1
    delta_max = 1.25
    delta     = 1.0

    xk = x0.astype(float).copy()
    Bk = np.eye(xk.size)
    gk = apGrad(f, xk)
    its  = 0

    while np.linalg.norm(gk, ord=np.inf) > tol and its < itmax:
        # 1) Paso Dog-Leg usando Bk
        pk = pDogLeg(Bk, gk, delta)

        # 2) Reducción real y predicha
        fk       = f(xk)
        fk_trial = f(xk + pk)
        red_real = fk - fk_trial
        red_pred = - (gk @ pk) - 0.5 * (pk @ Bk @ pk)

        # 3) Ratio rho
        rho = red_real / red_pred if red_pred > 0 else 0.0

        # 4) Ajuste de delta
        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and np.linalg.norm(pk) >= delta:
            delta = min(2 * delta, delta_max)

        # 5) Aceptar paso y actualizar BFGS
        if rho > eta:
            x_new = xk + pk
            g_new = apGrad(f, x_new)
            s     = x_new - xk
            y     = g_new - gk

            # Condición de curvatura: gamma^T s > 1e-3
            if y @ s > 1e-3:
                Bs = Bk @ s
                Bk = Bk \
                     - np.outer(Bs, Bs) / (s @ Bs) \
                     + np.outer(y,  y)  / (y @ s)

            xk, gk = x_new, g_new

        its += 1

    return xk, its


implementacion

In [ ]:


def rosenbrock_extended(x, c=100):
    """
    Función Rosenbrock extendida:
      f(x) = sum_{i=1}^{n/2} [ c*( x_{2i} - x_{2i-1}^2 )^2 + (1 - x_{2i-1})^2 ]

    Args:
        x (ndarray): vector de dimensión par n.
        c (float): constante de la función (por defecto 100).

    Returns:
        float: valor de f(x).
    """
    n = x.size
    if n % 2 != 0:
        raise ValueError("La dimensión debe ser par para Rosenbrock extendida.")
    f = 0.0
    for i in range(0, n, 2):
        xi, xi1 = x[i], x[i+1]
        f += c * (xi1 - xi**2)**2 + (1 - xi)**2
    return f

def make_x0(n):
    """
    Punto inicial para Rosenbrock:
      x0 = [–1.2, 1.0, –1.2, 1.0, …] de longitud n.
    """
    if n % 2 != 0:
        raise ValueError("n debe ser par para poder armar x0.")
    return np.tile([-1.2, 1.0], n//2)






simulacion mRCHess

In [ ]:
itmax = 100
c     = 100

for n in (2, 8, 32, 128):
    x0 = make_x0(n)
    tol, eta         = 1e-5, 0.1
    delta_max, delta = 1.25, 1.0
    xk = x0.copy()
    gk = apGrad(lambda x: rosenbrock_extended(x, c), xk)
    last5 = deque(maxlen=5)

    t0 = time.perf_counter()
    for k in range(1, itmax + 1):
        norm_inf = np.linalg.norm(gk, np.inf)
        fk       = rosenbrock_extended(xk, c)
        elapsed  = time.perf_counter() - t0

        last5.append((k, norm_inf, fk, elapsed))
        if norm_inf <= tol:
            break

        Bk      = apHess(lambda x: rosenbrock_extended(x, c), xk)
        pk      = pDogLeg(Bk, gk, delta)
        f_trial = rosenbrock_extended(xk + pk, c)
        red_real = fk - f_trial
        red_pred = - (gk @ pk) - 0.5 * (pk @ Bk @ pk)
        rho      = red_real / red_pred if red_pred > 0 else 0.0

        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and np.linalg.norm(pk) >= delta:
            delta = min(2 * delta, delta_max)

        if rho > eta:
            xk = xk + pk
            gk = apGrad(lambda x: rosenbrock_extended(x, c), xk)

    print(f"\nRosenbrock n={n}, últimas 5 iteraciones (mRCHess):")
    print("iter | norm_inf_grad |      f(x)      | time (s)")
    for it, norm_inf, fx, t in last5:
        print(f"{it:4d} | {norm_inf:13.6e} | {fx:13.6e} | {t:8.4f}")


Rosenbrock n=2, últimas 5 iteraciones (mRCHess):
iter | norm_inf_grad |      f(x)      | time (s)
  96 |  2.787533e+00 |  3.369368e+00 |   0.0219
  97 |  2.901429e+00 |  3.359859e+00 |   0.0221
  98 |  2.773565e+00 |  3.350337e+00 |   0.0223
  99 |  2.872062e+00 |  3.340710e+00 |   0.0224
 100 |  2.759154e+00 |  3.331065e+00 |   0.0226

Rosenbrock n=8, últimas 5 iteraciones (mRCHess):
iter | norm_inf_grad |      f(x)      | time (s)
  96 |  2.796884e+00 |  1.352922e+01 |   0.3214
  97 |  2.921369e+00 |  1.349150e+01 |   0.3245
  98 |  2.783207e+00 |  1.345374e+01 |   0.3275
  99 |  2.892280e+00 |  1.341556e+01 |   0.3306
 100 |  2.769103e+00 |  1.337732e+01 |   0.3338

Rosenbrock n=32, últimas 5 iteraciones (mRCHess):
iter | norm_inf_grad |      f(x)      | time (s)
  96 |  3.075759e+00 |  5.558299e+01 |   7.1575
  97 |  2.854172e+00 |  5.544164e+01 |   7.2983
  98 |  3.048637e+00 |  5.529855e+01 |   7.4335
  99 |  2.842156e+00 |  5.515556e+01 |   7.5658
 100 |  3.021193e+00 |  5.5010

Simulacion rcBFGS

In [ ]:
itmax = 500
c     = 100

for n in (2, 8, 32, 128):
    x0 = make_x0(n)
    # Inicializaciones
    tol, eta         = 1e-5, 0.1
    delta_max, delta = 1.25, 1.0
    xk               = x0.copy()
    gk               = apGrad(lambda x: rosenbrock_extended(x, c), xk)
    Bk               = np.eye(n)
    last5            = deque(maxlen=5)

    t0 = time.perf_counter()
    # Bucle principal
    for k in range(1, itmax+1):
        norm_inf = np.linalg.norm(gk, np.inf)
        f_k      = rosenbrock_extended(xk, c)
        err_inf  = np.linalg.norm(xk - np.ones(n), np.inf)
        elapsed  = time.perf_counter() - t0

        # Guardar la iteración
        last5.append((k, norm_inf, f_k, err_inf, elapsed))

        # Criterio de paro
        if norm_inf <= tol:
            break

        # Paso Dog-Leg
        pk = pDogLeg(Bk, gk, delta)

        # Evaluar reducción
        f_trial = rosenbrock_extended(xk + pk, c)
        red_real = f_k - f_trial
        red_pred = - (gk @ pk) - 0.5 * (pk @ Bk @ pk)
        rho      = red_real / red_pred if red_pred > 0 else 0.0

        # Ajuste de región
        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and np.linalg.norm(pk) >= delta:
            delta = min(2 * delta, delta_max)

        # Aceptar paso y actualizar BFGS
        if rho > eta:
            x_new = xk + pk
            g_new = apGrad(lambda x: rosenbrock_extended(x, c), x_new)
            s     = x_new - xk
            y     = g_new - gk
            if y @ s > 1e-3:
                Bs  = Bk @ s
                Bk  = Bk \
                    - np.outer(Bs, Bs)/(s @ Bs) \
                    + np.outer(y,  y)/(y @ s)
            xk, gk = x_new, g_new

    # Imprimir últimas 5 iteraciones
    print(f"\nRosenbrock n={n}, últimas 5 iteraciones (rcBFGS):")
    print("iter | norm_inf_grad |      f(x)      |  err_inf   | time (s)")
    for it, grad_norm, fx, err_inf, t in last5:
        print(f"{it:4d} | {grad_norm:13.6e} | {fx:13.6e} | {err_inf:11.6e} | {t:8.4f}")



Rosenbrock n=2, últimas 5 iteraciones (rcBFGS):
iter | norm_inf_grad |      f(x)      |  err_inf   | time (s)
  43 |  3.038613e-04 |  2.981855e-10 | 3.106383e-05 |   0.0189
  44 |  1.230193e-04 |  4.911047e-11 | 1.261480e-05 |   0.0191
  45 |  4.983434e-05 |  8.143284e-12 | 5.143970e-06 |   0.0192
  46 |  2.018345e-05 |  1.371752e-12 | 2.118538e-06 |   0.0194
  47 |  8.176210e-06 |  2.399629e-13 | 8.930367e-07 |   0.0195

Rosenbrock n=8, últimas 5 iteraciones (rcBFGS):
iter | norm_inf_grad |      f(x)      |  err_inf   | time (s)
 496 |  8.215327e-03 |  1.104107e-04 | 1.661762e-02 |   0.1627
 497 |  8.215327e-03 |  1.104107e-04 | 1.661762e-02 |   0.1631
 498 |  6.800404e-03 |  1.102639e-04 | 1.661119e-02 |   0.1635
 499 |  6.455933e-03 |  1.099822e-04 | 1.658931e-02 |   0.1637
 500 |  1.223407e-02 |  1.095266e-04 | 1.654898e-02 |   0.1639

Rosenbrock n=32, últimas 5 iteraciones (rcBFGS):
iter | norm_inf_grad |      f(x)      |  err_inf   | time (s)
 496 |  1.917378e-01 |  1.025348e-02

Simulacion rcSR1

In [ ]:
def rcSR1(f, x0, itmax, callback=None):

    # Parámetros
    tol    = 1e-5
    eta    = 1e-2    #aceptación del paso
    r_cond = 1e-3    #curvatura para actualizaciones SR1
    Delta_max = 1.25  # radio máximo
    Delta     = 1.0   # radio inicial

    # Inicialización
    xk, (f_val, gk) = x0.copy(), f(x0)
    n    = xk.size
    Bk   = np.eye(n)   # Hessiana
    Hk   = np.eye(n)
    its  = 0

    # principal
    while its < itmax and np.linalg.norm(gk, np.inf) > tol:
        # P1: dirección de descenso y radio
        pk = -Hk.dot(gk)
        norm_pk = np.linalg.norm(pk)
        if norm_pk > Delta:
            pk = (Delta / norm_pk) * pk

        # Evaluación
        x_trial = xk + pk
        f_new, g_new = f(x_trial)

        # P2: reducciones real y del modelo
        red_real = f_val - f_new
        red_pred = -(gk.dot(pk) + 0.5 * pk.dot(Bk.dot(pk)))
        rho = red_real / red_pred if red_pred != 0 else 0.0

        # P4–P5: ajuste del radio de confianza
        if rho > 0.75 and norm_pk > 0.8 * Delta:
            Delta = min(2.0 * Delta, Delta_max)
        elif rho < 0.1:
            Delta *= 0.5

        # P3: aceptación del paso
        if rho > eta:
            xk, f_val = x_trial, f_new
            its += 1

            # Actualización SR1 de Bk y Hk bajo condición de curvatura
            y  = g_new - gk
            Bs = Bk.dot(pk)
            u  = y - Bs
            if abs(u.dot(pk)) >= r_cond * np.linalg.norm(u) * norm_pk:
                Bk += np.outer(u, u) / u.dot(pk)

            v = pk - Hk.dot(y)
            if abs(v.dot(y)) >= r_cond * np.linalg.norm(v) * np.linalg.norm(y):
                Hk += np.outer(v, v) / v.dot(y)

            # Actualización del gradiente
            gk = g_new

            # Llamar callback tras cada iteración aceptada
            if callback:
                callback(its, xk, f_val, gk)

    return xk, its


In [ ]:


def simulate_rcSR1():
    dimensiones = [2, 8, 32, 128]
    itmax = 1000

    for n in dimensiones:
        x0 = np.tile(np.array([-1.2, 1.0]), n // 2)
        x_star = np.ones(n)
        historial = []
        t_start = time.time()

        def cb(k, xk, f_k, gk):
            t = time.time() - t_start
            historial.append({
                'k': k,
                '||∇f||inf': np.linalg.norm(gk, np.inf),
                'f(x)':   f_k,
                '||x−x*||inf': np.linalg.norm(xk - x_star, np.inf),
                't(s)':  t
            })
        rcSR1(lambda x: (rosenbrock_extended(x, c=100.0), apGrad(rosenbrock_extended, x)),
              x0, itmax, callback=cb)
        últimas5 = historial[-5:]
        print(f"\n=== rcSR1 para n = {n} ===")
        print(" k │ ∥∇f∥inf    │ f(x)        │ ∥x−x*∥inf    │ t (s)")
        print("───┼──────────┼─────────────┼────────────┼─────────")
        for row in últimas5:
            print(f"{row['k']:3d} │ "
                  f"{row['||∇f||inf']:<9.2e} │ "
                  f"{row['f(x)']:<11.2e} │ "
                  f"{row['||x−x*||inf']:<10.2e} │ "
                  f"{row['t(s)']:<7.2f}")

if __name__ == "__main__":
    simulate_rcSR1()




=== rcSR1 para n = 2 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
 42 │ 1.65e-02  │ 1.28e-06    │ 2.11e-03   │ 0.01   
 43 │ 4.21e-03  │ 4.84e-08    │ 3.87e-04   │ 0.01   
 44 │ 6.03e-05  │ 1.34e-11    │ 6.67e-06   │ 0.01   
 45 │ 1.34e-05  │ 1.34e-13    │ 3.06e-07   │ 0.01   
 46 │ 2.64e-08  │ 8.81e-16    │ 5.94e-08   │ 0.01   

=== rcSR1 para n = 8 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
 90 │ 8.10e-05  │ 2.04e-08    │ 2.03e-04   │ 0.02   
 91 │ 7.98e-05  │ 2.04e-08    │ 2.03e-04   │ 0.02   
 92 │ 6.21e-04  │ 1.03e-08    │ 1.41e-04   │ 0.02   
 93 │ 1.27e-04  │ 3.67e-11    │ 6.72e-06   │ 0.02   
 94 │ 5.98e-06  │ 1.44e-13    │ 5.08e-07   │ 0.02   

=== rcSR1 para n = 32 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
950 │ 3.62e-04  │ 2.05e-08    │ 1.23e-04   │ 1.57   
951 │ 4.22e-04  │ 1.35e-09    │ 3.0

In [ ]:
## Simulacion BFGSLiMem
##

import numpy as np
import time

def rcBFGSLiMem(f, x0, itmax, m, callback=None):

    # Parámetros de la region de confianza
    tol       = 1e-5
    eta       = 1e-2
    r_cond    = 1e-3
    delta_max = 1.0
    delta     = 0.5*delta_max

    # Inicialización
    xk, (f_val, gk) = x0.copy(), f(x0)
    its = 0
    s_list, y_list, rho_list = [], [], []

    # principal
    while its < itmax and np.linalg.norm(gk, np.inf) > tol:
        # 2 caminos
        q = gk.copy()
        alpha = [0.0]*len(s_list)
        for i in range(len(s_list)-1, -1, -1):
            alpha[i] = rho_list[i] * s_list[i].dot(q)
            q -= alpha[i] * y_list[i]
        if s_list:
            ys = y_list[-1].dot(s_list[-1])
            yy = y_list[-1].dot(y_list[-1])
            gamma = ys/yy
            r_vec = gamma*q
        else:
            r_vec = q.copy()
        for i in range(len(s_list)):
            beta = rho_list[i] * y_list[i].dot(r_vec)
            r_vec += (alpha[i] - beta)*s_list[i]
        pk = -r_vec
        norm_p = np.linalg.norm(pk)
        if norm_p > delta:
            pk = (delta/norm_p)*pk
        s = pk

        # reduccion
        x_trial = xk + s
        f_new, g_new = f(x_trial)
        y = g_new - gk
        redf = f_val - f_new
        redm = -(gk.dot(s) + 0.5*s.dot(y))
        rho = redf/(redm if redm!=0 else 1e-16)

        # ajuste de radio
        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and abs(norm_p - delta) < 1e-12:
            delta = min(2*delta, delta_max)

        # aceptar
        if rho > eta:
            xk, f_val, gk = x_trial, f_new, g_new
            its += 1

            # actualizar memoria
            sy = y.dot(s)
            if sy > r_cond * np.linalg.norm(s) * np.linalg.norm(y):
                if len(s_list) == m:
                    s_list.pop(0)
                    y_list.pop(0)
                    rho_list.pop(0)
                s_list.append(s)
                y_list.append(y)
                rho_list.append(1.0/sy)

            # callback si se acepta el paso
            if callback:
                callback(its, xk, f_val, gk)

    return xk, its



def simulate_rcBFGSLiMem():
    dimensiones = [2, 8, 32, 128]
    itmax = 2000
    m = 5

    for n in dimensiones:
        x0 = np.tile(np.array([-1.2, 1.0]), n//2)
        x_star = np.ones(n)
        historial = []
        t_start = time.time()

        def cb(k, xk, f_k, gk):
            t = time.time() - t_start
            historial.append({
                'k': k,
                '||∇f||∞': np.linalg.norm(gk, np.inf),
                'f(x)':   f_k,
                '||x−x*||∞': np.linalg.norm(xk - x_star, np.inf),
                't(s)':  t
            })

        # Ejecuta el optimizador con callback
        rcBFGSLiMem(lambda x: (rosenbrock_extended(x, c=100.0), apGrad(rosenbrock_extended, x)),
            x0, itmax, m, callback=cb)


        # Imprime últimas 5 iteraciones aceptadas
        últimas5 = historial[-5:]
        print(f"\n=== rcBFGSLiMem para n = {n} ===")
        print(" k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)")
        print("───┼──────────┼─────────────┼────────────┼─────────")
        for row in últimas5:
            print(f"{row['k']:3d} │ "
                  f"{row['||∇f||∞']:<9.2e} │ "
                  f"{row['f(x)']:<11.2e} │ "
                  f"{row['||x−x*||∞']:<10.2e} │ "
                  f"{row['t(s)']:<7.2f}")


if __name__ == "__main__":
    simulate_rcBFGSLiMem()




=== rcBFGSLiMem para n = 2 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
 38 │ 1.91e-02  │ 2.93e-05    │ 1.08e-02   │ 0.01   
 39 │ 4.47e-03  │ 1.26e-06    │ 2.23e-03   │ 0.01   
 40 │ 2.29e-03  │ 3.68e-09    │ 4.02e-05   │ 0.01   
 41 │ 7.88e-04  │ 3.90e-10    │ 3.38e-06   │ 0.01   
 42 │ 2.33e-07  │ 4.85e-17    │ 1.09e-08   │ 0.01   

=== rcBFGSLiMem para n = 8 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
 77 │ 2.71e-05  │ 2.24e-10    │ 1.68e-05   │ 0.03   
 78 │ 4.37e-05  │ 2.10e-10    │ 1.85e-05   │ 0.03   
 79 │ 4.83e-05  │ 2.03e-10    │ 2.05e-05   │ 0.03   
 80 │ 9.59e-05  │ 2.05e-10    │ 2.14e-05   │ 0.03   
 81 │ 8.34e-06  │ 1.99e-10    │ 2.16e-05   │ 0.03   

=== rcBFGSLiMem para n = 32 ===
 k │ ∥∇f∥∞    │ f(x)        │ ∥x−x*∥∞    │ t (s)
───┼──────────┼─────────────┼────────────┼─────────
 60 │ 3.59e-05  │ 4.25e-11    │ 4.37e-06   │ 0.14   
 61 │ 1.55e-05  │

In [ ]:
import numpy as np
import time



def dixmaana(x):
        #α = 1, β = 0, γ = 0.125, δ = 0.125
        #k1 = 2, k2 = 0, k3 = 2, k4 = 0
        #m = n // 3
    x = np.asarray(x, dtype=float)
    n = x.size
    if n % 3 != 0:
        raise ValueError("n debe ser divisible entre 3")

    m = n // 3

    # indices
    i_vals = np.arange(1, n + 1) / n

    # Parametros
    alpha, beta, gamma, delta = 1, 0, 0.125, 0.125
    k1, k2, k3, k4 = 2, 0, 2, 0

    # Calcular terminos
    term1 = alpha * np.sum(x**2 * i_vals**k1)
    term2 = 0
    i_m = np.arange(1, 2*m + 1) / n
    x_first_2m = x[:2*m]
    x_m_to_3m = x[m:3*m]
    term3 = gamma * np.sum(x_first_2m**2 * x_m_to_3m**4 * i_m**k3)

    i_m_shift = np.arange(1, m + 1) / n
    term4 = delta * np.sum(x[:m] * x[2*m:3*m] * i_m_shift**k4)

    return 1 + term1 + term2 + term3 + term4


def dixmaana_with_grad(x):
    """
    Returns both function value and gradient for DIXMAANA.
    """
    f_val = dixmaana(x)
    grad = apGrad(dixmaana, x)
    return f_val, grad


def rcBFGSLiMem(f, x0, itmax, m):

    tol = 1e-5            # Convergence tolerance
    eta = 1e-2            # Acceptance threshold
    r_cond = 1e-3         # Curvature condition threshold
    delta_max = 1.25      # Maximum trust region radius
    delta = 1.0           # Initial trust region radius

    xk = x0.copy()
    f_val, gk = f(x0)

    s_list, y_list, rho_list = [], [], []
    its = 0

    t_start = time.time()

    while its < itmax and np.linalg.norm(gk, np.inf) > tol:
        # Two-loop recursion for direction computation
        q = gk.copy()
        alpha = [0.0] * len(s_list)

        # First loop (backward)
        for i in range(len(s_list) - 1, -1, -1):
            alpha[i] = rho_list[i] * s_list[i].dot(q)
            q -= alpha[i] * y_list[i]

        # Initial Hessian approximation
        if s_list:
            ys = y_list[-1].dot(s_list[-1])
            yy = y_list[-1].dot(y_list[-1])
            gamma = ys / yy
            r_vec = gamma * q
        else:
            r_vec = q.copy()

        # Second loop (forward)
        for i in range(len(s_list)):
            beta = rho_list[i] * y_list[i].dot(r_vec)
            r_vec += (alpha[i] - beta) * s_list[i]

        # Search direction is negative of the result
        pk = -r_vec

        # Enforce trust region constraint
        norm_pk = np.linalg.norm(pk)
        if norm_pk > delta:
            pk *= delta / norm_pk

        # Trial step
        x_trial = xk + pk
        f_new, g_new = f(x_trial)

        # Compute actual and predicted reduction
        y = g_new - gk
        red_real = f_val - f_new
        red_pred = -(gk.dot(pk) + 0.5 * pk.dot(y))

        # Compute ratio
        rho = red_real / red_pred if red_pred != 0 else 0.0

        # Update trust region radius
        if rho < 0.25:
            delta *= 0.25
        elif rho > 0.75 and abs(norm_pk - delta) < 1e-12:
            delta = min(2 * delta, delta_max)

        # Accept or reject step
        if rho > eta:
            # Accept step
            xk, f_val, gk = x_trial, f_new, g_new
            its += 1

            # Update L-BFGS memory
            sy = y.dot(pk)
            if sy > r_cond * np.linalg.norm(pk) * np.linalg.norm(y):
                if len(s_list) == m:
                    s_list.pop(0)
                    y_list.pop(0)
                    rho_list.pop(0)
                s_list.append(pk)
                y_list.append(y)
                rho_list.append(1.0 / sy)

    total_time = time.time() - t_start

    return xk, its, np.linalg.norm(gk, np.inf), f_val, total_time


def simulate_rcBFGSLiMem():
    """Simulation of the rcBFGSLiMem method on the DIXMAANA function."""
    dimensiones = [240, 960]
    valores_m = [1, 3, 5, 17, 29]
    itmax = 200

    for n in dimensiones:
        x0 = np.ones(n)
        resultados = []

        for m in valores_m:
            # Use a wrapper function that combines dixmaana and its gradient
            f_wrapper = lambda x: (dixmaana(x), apGrad(dixmaana, x))

            # Run the optimization
            x_opt, its, grad_norm_inf, f_val, tiempo = rcBFGSLiMem(f_wrapper, x0, itmax, m)

            # Store results
            resultados.append([m, its, grad_norm_inf, f_val, tiempo])

        # Print results table
        print(f"\n=== rcBFGSLiMem para n = {n} ===")
        print(" m  │ iteraciones │ ∥∇f∥∞    │ f(x)        │ t (s)")
        print("────┼────────────┼──────────┼─────────────┼─────────")
        for row in resultados:
            print(f"{row[0]:3d} │ {row[1]:12d} │ {row[2]:<9.2e} │ {row[3]:<11.2e} │ {row[4]:<7.2f}")


if __name__ == "__main__":
    simulate_rcBFGSLiMem()


ValueError: not enough values to unpack (expected 5, got 2)